# GPU benchmarks — Financial Fundamentals Analysis

Measures on a Colab GPU what the README currently only reports for CPU.

The open question this exists to answer: **does the parallel fan-out help on a GPU?** On CPU the answer turned out to be counterintuitive — naive fan-out ran 9% *slower* than sequential, because torch takes roughly one intra-op thread per core *per model* and three concurrent branches oversubscribe the machine. Capping threads at cores ÷ branches turned that into a 1.13x speedup.

On a GPU the tradeoff is different in kind: the branches share one device instead of competing for CPU threads, and the GPU serializes much of the work regardless. Whether concurrency wins anything there has **not** been measured — the README says the thread cap is skipped on GPU, which is reasoning, not evidence. This notebook produces the evidence.

It also re-runs the CPU sweep *on this same machine*, because comparing a Colab T4 against a 12-core laptop would confound the hardware with the topology.

---

### Before you run

1. **Set a GPU runtime**: Runtime → Change runtime type → Hardware accelerator → **GPU** (T4 is fine), then Save.
2. **Push your local commits.** This notebook clones from GitHub. Any work still sitting on your laptop will not be in the clone — step 3 checks for this and tells you if the clone is stale.

## 1. Confirm the GPU

In [ ]:
!nvidia-smi

import torch
print("\nCUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU. Runtime -> Change runtime type -> Hardware accelerator -> GPU")

## 2. Clone the repository

In [ ]:
REPO = "https://github.com/abhinaba01/fundamental-financial-analysis.git"
BRANCH = "main"

import os

if not os.path.exists("fundamental-financial-analysis"):
    !git clone -q --branch {BRANCH} {REPO}

%cd fundamental-financial-analysis
!git log --oneline -5

## 3. Check the clone is current

The benchmark needs two things that only exist in recent commits: the parallel fan-out in `builder.py`, and the `--gpu` flag on the benchmark script. If either is missing, the clone predates them and everything below would silently measure the old sequential pipeline instead.

In [ ]:
from pathlib import Path

checks = {
    "parallel fan-out (src/graph/builder.py)":
        "PARALLEL_ANALYSIS_NODES" in Path("src/graph/builder.py").read_text(),
    "--gpu flag (scripts/benchmark_parallel.py)":
        Path("scripts/benchmark_parallel.py").exists()
        and "--gpu" in Path("scripts/benchmark_parallel.py").read_text(),
    "split retrieve/generate nodes (src/agents/rag_agent.py)":
        "def retrieve" in Path("src/agents/rag_agent.py").read_text(),
}

for label, ok in checks.items():
    print(f"  {'OK  ' if ok else 'MISSING'}  {label}")

if not all(checks.values()):
    raise SystemExit(
        "\nThis clone predates the work being benchmarked.\n"
        "Push your local commits, then delete the cloned folder and re-run step 2:\n"
        "    git push origin main\n"
    )

print("\nClone is current.")

## 4. Install

Colab already ships a CUDA-enabled torch, and the `>=2.0.0` constraint is satisfied, so this will not reinstall it. The `[eval]` extra pins `datasets<4.0` — required because `takala/financial_phrasebank` is a legacy loading-script dataset and 4.0 removed script execution.

If pip asks you to restart the runtime, do it, then re-run from step 2 (`%cd` is lost on restart).

In [ ]:
!pip install -q -e ".[dev,eval]"
!python -m spacy download en_core_web_sm -q

import torch
print("torch", torch.__version__, "| CUDA", torch.cuda.is_available())

In [ ]:
# Pull model weights now so download time never lands inside a timed run.
!python scripts/download_models.py

## 5. The measurement: fan-out on GPU

Builds two graphs over the *same* loaded models — one fanning NER / sentiment / KPI out from START, one chaining them — and times both on the same document.

Timing is fenced with `torch.cuda.synchronize()` at both ends of every run. CUDA kernel launches are asynchronous, so without that fence the timer measures how fast Python queued the work rather than how long the GPU took, and reports an impossibly large speedup.

In [ ]:
gpu_out = !python scripts/benchmark_parallel.py --gpu --runs 5
print("\n".join(gpu_out))

## 6. CPU baseline on this same machine

Needed for a fair comparison — the README's CPU numbers come from a 12-core laptop, and a Colab VM typically has only 2 vCPUs. That difference matters for this specific result: with 2 cores, `cores ÷ branches` rounds down to 1 thread per branch, so the thread-capping trick that won on a 12-core box may not reproduce here at all.

Expect these cells to be slow (several minutes each) — that slowness is itself the point of the GPU comparison.

In [ ]:
import os
print("logical cores on this VM:", os.cpu_count())

cpu_default = !python scripts/benchmark_parallel.py --runs 3
print("\n".join(cpu_default))

In [ ]:
# The capped-thread variant. On a 2-vCPU VM this may be identical to the
# default - report whatever it actually does rather than the laptop's result.
capped = max(1, (os.cpu_count() or 1) // 3)
print(f"capping intra-op threads at {capped}")

cpu_capped = !python scripts/benchmark_parallel.py --runs 3 --torch-threads {capped}
print("\n".join(cpu_capped))

## 7. End-to-end pipeline on GPU

The whole thing, on a real document, so the GPU number is a wall-clock figure for the actual product rather than only the analysis phase.

Without `OPENAI_API_KEY` the RAG agent falls back to extractive synthesis and the pipeline still runs end to end — the `reasoning` fields are stitched chunk text instead of a generated answer. Set the key in Colab's Secrets (🔑 in the sidebar) if you want real generation.

In [ ]:
import os

try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
    print("Loaded OPENAI_API_KEY from Colab Secrets.")
except Exception:
    print("No OPENAI_API_KEY - RAG will use extractive synthesis. Pipeline still runs.")

In [ ]:
import time

start = time.perf_counter()
!python -m src.main \
  --document data/samples/AAPL_10K.pdf \
  --query "What are the primary risk factors?" \
  --output aapl_gpu_report.json
e2e_gpu = time.perf_counter() - start
print(f"\nend-to-end on GPU: {e2e_gpu:.1f}s")

## 8. Sentiment benchmark on GPU

Financial PhraseBank, all 2,264 samples. This took about 10 minutes on CPU; on a T4 it should be well under a minute, which is the clearest single illustration of what the GPU buys for bulk evaluation.

It should reproduce **accuracy 0.972 / macro-F1 0.963**. If it does not, something differs between the environments and is worth chasing — the metric itself is deterministic.

Reminder on what this number means: `ProsusAI/finbert` was fine-tuned on Financial PhraseBank, so this scores the model against its own training data. It checks the harness, not the model's generalization.

In [ ]:
!python scripts/prepare_eval_datasets.py --dataset phrasebank

In [ ]:
import time

start = time.perf_counter()
!python -m evaluation.eval_sentiment \
  --test-set data/eval/phrasebank_test.json \
  --run-agent \
  --output data/eval/results_phrasebank_gpu.json
sentiment_secs = time.perf_counter() - start
print(f"\nsentiment benchmark wall clock: {sentiment_secs:.1f}s  (CPU reference: ~600s)")

## 9. Summary

Parses the runs above into one table. Paste it into the README's Measured Results section, replacing the placeholder GPU row.

In [ ]:
import re


def parse(lines):
    """Pull the median timings out of a benchmark run's output."""
    text = "\n".join(lines)
    seq = re.search(r"sequential \(median\):\s*([\d.]+)s", text)
    par = re.search(r"parallel\s+\(median\):\s*([\d.]+)s", text)
    if not (seq and par):
        return None
    return float(seq.group(1)), float(par.group(1))


rows = []
for label, lines in [
    ("GPU", gpu_out),
    (f"CPU, default threads", cpu_default),
    (f"CPU, {capped} thread(s)/branch", cpu_capped),
]:
    parsed = parse(lines)
    if parsed:
        seq, par = parsed
        rows.append((label, seq, par, seq / par))

print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'n/a'}")
print(f"vCPUs: {os.cpu_count()}\n")
print("| Device | Sequential | Parallel | Speedup |")
print("|--------|-----------:|---------:|--------:|")
for label, seq, par, ratio in rows:
    print(f"| {label} | {seq:.1f}s | {par:.1f}s | {ratio:.2f}x |")

print(f"\nEnd-to-end on GPU (AAPL 10-K): {e2e_gpu:.1f}s")
print(f"Sentiment benchmark on GPU (n=2264): {sentiment_secs:.1f}s vs ~600s on CPU")

## Notes

- **A speedup below 1.00x on GPU is a real result, not a failed run.** The three branches share one device; if the GPU is already saturated by one model, running three concurrently cannot overlap much. Report whatever comes out.
- **Colab VMs vary.** vCPU count and GPU model differ between sessions, so record both (the summary cell prints them) alongside any number you quote.
- **Free-tier runtimes get recycled.** Everything here — the clone, the model cache, `data/vector_store/` — disappears when the runtime disconnects.
- The `--torch-threads` flag is CPU-only. It is ignored for the GPU run, where the models are not competing for intra-op threads.